# BE 5210: Final Project 

Kevin Han, Noah Son, Sophia Fu

# Project Overview

This final project involves predicting finger flexion using intracranial EEG (ECoG) in three human subjects. The data and problem framing come from the 4th BCI Competition (Miller et al. 2008). For the details of the problem, experimental protocol, data, and evaluation, please see the original 4th BCI Competition documentation (included as separate document). The remainder of the current document details your deliverables for part 1 of the project.


# Data Cleaning Pipeline

This section preprocesses the raw ECoG and dataglove recordings using the wavelet-based pipeline defined in `data_processing.py`. The steps are:

1. **Reshape** – transpose from `(samples, channels)` to `(channels, samples)`
2. **Normalize** – z-score per channel + common average referencing
3. **Bandpass filter** – keep 40–300 Hz; remove powerline harmonics (60 Hz)
4. **Morlet wavelet spectrograms** – 40 log-spaced frequencies → shape `(channels, 40, time)`
5. **Downsample** – 1000 Hz → 100 Hz
6. **Time-delay alignment** – shift by 200 ms to account for neural-to-motor lag
7. **Scale** – `MinMaxScaler` on finger flex; `RobustScaler` on ECoG spectrograms
8. **Save** – write `.npy` arrays to `cleaned_data/`

In [1]:
import sys, pathlib, pickle
import numpy as np
import scipy.io

PROJECT_ROOT = pathlib.Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from data_processing import TIME_DELAY_SECS

RAW_DATA_DIR   = PROJECT_ROOT / "raw_data"
CLEAN_DATA_DIR = PROJECT_ROOT / "cleaned_data"
CLEAN_DATA_DIR.mkdir(exist_ok=True)

## Load Raw Data

`raw_training_data.mat` contains three subjects:
- `train_ecog`: list of `(300 000, channels)` arrays at 1000 Hz
- `train_dg`:   list of `(300 000, 5)` finger-flex arrays at 1000 Hz

`leaderboard_data.mat` contains `leaderboard_ecog` (no labels) for final predictions.

In [2]:
train_mat = scipy.io.loadmat(RAW_DATA_DIR / "raw_training_data.mat")
lead_mat  = scipy.io.loadmat(RAW_DATA_DIR / "leaderboard_data.mat")

N_SUBJECTS = 3

train_ecog = [train_mat["train_ecog"][i, 0].astype("float64") for i in range(N_SUBJECTS)]
train_dg   = [train_mat["train_dg"][i, 0].astype("float64")   for i in range(N_SUBJECTS)]
lead_ecog  = [lead_mat["leaderboard_ecog"][i, 0].astype("float64") for i in range(N_SUBJECTS)]

for i in range(N_SUBJECTS):
    print(f"Subject {i+1}:  ecog={train_ecog[i].shape}  dg={train_dg[i].shape}  "
          f"leaderboard_ecog={lead_ecog[i].shape}")

Subject 1:  ecog=(300000, 62)  dg=(300000, 5)  leaderboard_ecog=(147500, 62)
Subject 2:  ecog=(300000, 48)  dg=(300000, 5)  leaderboard_ecog=(147500, 48)
Subject 3:  ecog=(300000, 64)  dg=(300000, 5)  leaderboard_ecog=(147500, 64)


## Run Preprocessing Pipeline

Each subject's training data and leaderboard data are processed through the same pipeline. Scalers are fit **only on the training set** and applied to the leaderboard set to prevent data leakage.

In [3]:
from data_processing import (
    reshape_ecog, normalize_ecog, filter_ecog,
    compute_spectrograms, downsample_spectrograms,
    downsample_fingerflex, crop_for_time_delay,
    scale_fingerflex, scale_ecog,
)

cleaned = {}  # will hold results per subject

for subj in range(N_SUBJECTS):
    print(f"\n{'='*50}")
    print(f"  Subject {subj + 1}")
    print(f"{'='*50}")

    # ------------------------------------------------------------------ #
    # Training ECoG
    # ------------------------------------------------------------------ #
    print("  [Train ECoG] reshaping + normalizing + filtering...")
    ecog_train = filter_ecog(
        normalize_ecog(reshape_ecog(train_ecog[subj]))
    )

    print("  [Train ECoG] computing Morlet spectrograms...")
    specs_train = compute_spectrograms(ecog_train)
    specs_train = downsample_spectrograms(specs_train)

    # ------------------------------------------------------------------ #
    # Training finger flex
    # ------------------------------------------------------------------ #
    print("  [Train DG]   downsampling finger flex...")
    ff_train = downsample_fingerflex(reshape_ecog(train_dg[subj]))

    # ------------------------------------------------------------------ #
    # Time-delay alignment
    # ------------------------------------------------------------------ #
    print(f"  Applying {TIME_DELAY_SECS*1000:.0f} ms time-delay alignment...")
    ff_train, specs_train = crop_for_time_delay(ff_train, specs_train)

    # ------------------------------------------------------------------ #
    # Leaderboard ECoG (no labels)
    # ------------------------------------------------------------------ #
    print("  [Lead ECoG]  reshaping + normalizing + filtering...")
    ecog_lead = filter_ecog(
        normalize_ecog(reshape_ecog(lead_ecog[subj]))
    )

    print("  [Lead ECoG]  computing Morlet spectrograms...")
    specs_lead = downsample_spectrograms(compute_spectrograms(ecog_lead))

    # ------------------------------------------------------------------ #
    # Scale — fit on train, apply to both
    # ------------------------------------------------------------------ #
    print("  Scaling...")
    _, ff_train_scaled = scale_fingerflex(ff_train)
    ecog_scaler, specs_train_scaled, specs_lead_scaled = scale_ecog(
        specs_train, specs_lead
    )

    cleaned[subj] = {
        "specs_train": specs_train_scaled,   # (channels, 40, T_train)
        "ff_train":    ff_train_scaled,       # (5, T_train)
        "specs_lead":  specs_lead_scaled,     # (channels, 40, T_lead)
        "ecog_scaler": ecog_scaler,
    }

    print(f"  Train ECoG:  {specs_train_scaled.shape}")
    print(f"  Train DG:    {ff_train_scaled.shape}")
    print(f"  Lead ECoG:   {specs_lead_scaled.shape}")


  Subject 1
  [Train ECoG] reshaping + normalizing + filtering...
Setting up band-pass filter from 40 - 3e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 40.00
- Lower transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 35.00 Hz)
- Upper passband edge: 300.00 Hz
- Upper transition bandwidth: 75.00 Hz (-6 dB cutoff frequency: 337.50 Hz)
- Filter length: 331 samples (0.331 s)

Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 6601 samples (6.601 s)

  [Train ECoG] computing Morlet 

## Save Cleaned Data

Arrays are saved to `cleaned_data/subj{N}/` as `.npy` files. The scalers are saved with `pickle` so they can be reused for inverse-transforming predictions.

In [5]:
import pickle

for subj, data in cleaned.items():
    subj_dir = CLEAN_DATA_DIR / f"subj{subj + 1}"
    subj_dir.mkdir(exist_ok=True)

    np.save(subj_dir / "specs_train.npy", data["specs_train"])
    np.save(subj_dir / "ff_train.npy",    data["ff_train"])
    np.save(subj_dir / "specs_lead.npy",  data["specs_lead"])

    with open(subj_dir / "ff_train.pkl",   "wb") as f:
        pickle.dump(data["ff_train"], f)
    with open(subj_dir / "ecog_scaler.pkl", "wb") as f:
        pickle.dump(data["ecog_scaler"], f)

    print(f"Subject {subj + 1} saved to {subj_dir}")

Subject 1 saved to /Users/home/BE5210Homework/BE5210FinalProject/cleaned_data/subj1
Subject 2 saved to /Users/home/BE5210Homework/BE5210FinalProject/cleaned_data/subj2
Subject 3 saved to /Users/home/BE5210Homework/BE5210FinalProject/cleaned_data/subj3


## Verify Saved Data

Quick sanity check — reload and confirm shapes match expectations.

In [6]:
for subj_num in range(1, N_SUBJECTS + 1):
    subj_dir = CLEAN_DATA_DIR / f"subj{subj_num}"
    specs = np.load(subj_dir / "specs_train.npy")
    ff    = np.load(subj_dir / "ff_train.npy")
    lead  = np.load(subj_dir / "specs_lead.npy")
    print(f"Subject {subj_num}:")
    print(f"  specs_train : {specs.shape}  (channels, freqs, time)")
    print(f"  ff_train    : {ff.shape}   (fingers, time)")
    print(f"  specs_lead  : {lead.shape}")

Subject 1:
  specs_train : (62, 40, 29980)  (channels, freqs, time)
  ff_train    : (5, 29980)   (fingers, time)
  specs_lead  : (62, 40, 14750)
Subject 2:
  specs_train : (48, 40, 29980)  (channels, freqs, time)
  ff_train    : (5, 29980)   (fingers, time)
  specs_lead  : (48, 40, 14750)
Subject 3:
  specs_train : (64, 40, 29980)  (channels, freqs, time)
  ff_train    : (5, 29980)   (fingers, time)
  specs_lead  : (64, 40, 14750)
